# Kaggle – Data on the top
Tu profe ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Métrica: RMSE

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

Donde $y_i$ es el valor real y $\hat{y}_i$ es el valor predicho. **Cuanto menor, mejor.**

---
# PARTE 1: Entrenamiento del modelo

## 1. Librerías

In [2]:
import numpy as np
import pandas as pd
import re 
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

## 2. Datos

In [3]:
df = pd.read_csv('./data/train.csv', encoding='latin-1')

### 2.1 Exploración de los datos

In [4]:
df.head()


,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
0,755,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.86kg,539.00
1,618,Dell,Inspiron 7559,Gaming,15.6,Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16GB,1TB HDD,Nvidia GeForce GTX 960<U+039C>,Windows 10,2.59kg,879.01
2,909,HP,ProBook 450,Notebook,15.6,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,8GB,1TB HDD,Nvidia GeForce 930MX,Windows 10,2.04kg,900.00
3,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
4,286,Dell,Inspiron 3567,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,AMD Radeon R5 M430,Linux,2.25kg,428.00


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 912 entries, 0 to 911
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   laptop_ID         912 non-null    int64  
 1   Company           912 non-null    str    
 2   Product           912 non-null    str    
 3   TypeName          912 non-null    str    
 4   Inches            912 non-null    float64
 5   ScreenResolution  912 non-null    str    
 6   Cpu               912 non-null    str    
 7   Ram               912 non-null    str    
 8   Memory            912 non-null    str    
 9   Gpu               912 non-null    str    
 10  OpSys             912 non-null    str    
 11  Weight            912 non-null    str    
 12  Price_in_euros    912 non-null    float64
dtypes: float64(2), int64(1), str(10)
memory usage: 92.8 KB


In [6]:
for col in df.columns:
    print(df[col].value_counts())

laptop_ID
755     1
618     1
909     1
2       1
286     1
       ..
28      1
1160    1
78      1
23      1
229     1
Name: count, Length: 912, dtype: int64
Company
Lenovo       202
Dell         197
HP           194
Asus         121
Acer          74
MSI           37
Toshiba       34
Apple         17
Razer          6
Mediacom       6
Samsung        5
Microsoft      5
Xiaomi         3
Chuwi          2
Huawei         2
Google         2
Vero           2
Fujitsu        2
LG             1
Name: count, dtype: int64
Product
XPS 13                               23
Inspiron 3567                        22
Legion Y520-15IKBN                   15
Vostro 3568                          14
250 G6                               13
                                     ..
Aspire A515-51G-37JS                  1
110-15ACL (A6-7310/4GB/500GB/W10)     1
Rog G752VL-UH71T                      1
Vivobook Max                          1
GL62M 7RD                             1
Name: count, Length: 480, dtype: int

In [7]:
df.describe()

,laptop_ID,Inches,Price_in_euros
count,912.000000,912.000000,912.000000
mean,650.312500,14.981579,1111.724090
std,382.727748,1.436719,687.959172
min,2.000000,10.100000,174.000000
25%,324.750000,14.000000,589.000000
50%,636.500000,15.600000,978.000000
75%,982.250000,15.600000,1483.942500
max,1320.000000,18.400000,6099.000000


### 2.2 Definir X e y


In [8]:
df.rename(columns={'Price_in_euros': 'target'}, inplace=True)

### 2.3 Dividir en train y test

In [9]:
x_train, x_test, y_train, y_test = train_test_split(df.drop(columns=['target']), df['target'], test_size=0.2, random_state=42)


## 3. Procesado de datos

> 🚨 **Data leakage:** si usas un scaler, haz **`.fit()` SOLO sobre `X_train`** y luego aplica `.transform()` sobre `X_train` y `X_test` por separado.
>
> Recuerda también que **todo lo que hagas aquí deberás replicarlo después en `test.csv`** (sección 6).

In [10]:
import re
import pandas as pd

def advanced_cleaning(input_df):
    df_clean = input_df.copy()
    
    #Limpieza básica de unidades
    df_clean["Weight"] = df_clean["Weight"].str.replace("kg", "").astype(float)
    df_clean["Ram_GB"] = df_clean["Ram"].str.replace("GB", "").astype(int)
    if "Inches" in df_clean.columns:
        df_clean["Inches"] = df_clean["Inches"].astype(float)
        
    #Características de la Pantalla
    df_clean["Ops_IPS"] = df_clean["ScreenResolution"].str.contains("IPS").astype(int)
    df_clean["Ops_FullHD"] = df_clean["ScreenResolution"].str.contains("Full HD").astype(int)
    
    #Procesador
    df_clean["Cpu_GHz"] = df_clean["Cpu"].str.extract(r'(\d+(?:\.\d+)?)GHz').astype(float)
    df_clean["Cpu_Brand"] = df_clean["Cpu"].apply(lambda x: "Intel Core i7" if "Core i7" in x 
                                                  else "Intel Core i5" if "Core i5" in x 
                                                  else "Intel Core i3" if "Core i3" in x 
                                                  else "Intel Celeron/Pentium" if ("Celeron" in x or "Pentium" in x)
                                                  else "AMD" if "AMD" in x else "Otros Intel")

    #Almacenamiento
    def extract_storage(text, type_storage):
        text = text.replace("GB", "").replace("TB", "000")
        match = re.search(r'(\d+)\s*' + type_storage, text)
        return int(match.group(1)) if match else 0

    df_clean["SSD_GB"] = df_clean["Memory"].apply(lambda x: extract_storage(x, "SSD"))
    df_clean["HDD_GB"] = df_clean["Memory"].apply(lambda x: extract_storage(x, "HDD"))

    #Tarjeta Gráfica
    df_clean["Gpu_Brand"] = df_clean["Gpu"].apply(lambda x: x.split()[0])
    
    #Índices y eliminación de columnas originales de texto
    df_clean.set_index('laptop_ID', inplace=True)
    cols_to_drop = ['Ram', 'Memory', 'Cpu', 'Gpu', 'ScreenResolution', 'Product']
    df_clean.drop(columns=cols_to_drop, inplace=True, errors='ignore')
    
    return df_clean

#train y test
X_train_clean = advanced_cleaning(x_train)
X_test_clean = advanced_cleaning(x_test)

#One-Hot Encoding unificado
categorical_features = ['Company', 'TypeName', 'OpSys', 'Cpu_Brand', 'Gpu_Brand']
combined = pd.concat([X_train_clean, X_test_clean], axis=0, keys=['train', 'test'])
combined_encoded = pd.get_dummies(combined, columns=categorical_features, drop_first=True)

X_train_cat = combined_encoded.xs('train')
X_test_cat = combined_encoded.xs('test')

## 4. Modelado

### 4.1 Entrenamiento

In [11]:
rnd_forest = RandomForestRegressor(
    n_estimators=150, 
    max_depth=9, 
    min_samples_split=5, 
    random_state=10
)
rnd_forest.fit(X_train_cat, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",150
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",9
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",10
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of t

### 4.2 Métricas

Recuerda que en la competición se evalúa con **RMSE**.

In [12]:
y_pred_local = rnd_forest.predict(X_test_cat)
rmse_local = root_mean_squared_error(y_test, y_pred_local)
print(f"RMSE: {rmse_local:.2f} €")

RMSE: 344.21 €


### 4.3 Optimización (up to you 🫰🏻)

In [ ]:
# Tu código aquí


## 5. Reentrenamiento sobre todos los datos de `train.csv`

Una vez afinado el modelo, reentrenamos con **todos** los datos disponibles antes de predecir sobre `test.csv`.

> ¿Por qué? El split anterior era solo para validar localmente. Para la submission final queremos aprovechar el 100% de los datos de entrenamiento.

In [13]:
#Limpieza del dataset completo
df_full_clean = advanced_cleaning(df)
y_full_train = df['target']
if 'target' in df_full_clean.columns:
    df_full_clean.drop(columns=['target'], inplace=True)

#Carga y limpieza
X_pred_raw = pd.read_csv('./data/test.csv', encoding='latin-1')
X_pred_clean = advanced_cleaning(X_pred_raw)

#Codificación conjunta
combined_full = pd.concat([df_full_clean, X_pred_clean], axis=0, keys=['train', 'test'])
combined_full_encoded = pd.get_dummies(combined_full, columns=categorical_features, drop_first=True)

X_full_train = combined_full_encoded.xs('train')
X_pred_final = combined_full_encoded.xs('test')

#Entrenamiento final con los parámetros ganadores
final_model = RandomForestRegressor(
    n_estimators=150, 
    max_depth=9, 
    min_samples_split=5, 
    random_state=10
)
final_model.fit(X_full_train, y_full_train)
print("Modelo entrenado")

Modelo entrenado


---
# PARTE 2: Predicción y submission

Una vez tengas el modelo listo, toca predecir sobre `test.csv` y generar el archivo de submission.

## 6. Carga los datos de `test.csv`

In [14]:
X_pred_raw.head()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
0,209,Lenovo,Legion Y520-15IKBN,Gaming,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD,Nvidia GeForce GTX 1060,No OS,2.4kg
1,1281,Acer,Aspire ES1-531,Notebook,15.6,1366x768,Intel Celeron Dual Core N3060 1.6GHz,4GB,500GB HDD,Intel HD Graphics 400,Linux,2.4kg
2,1168,Lenovo,V110-15ISK (i3-6006U/4GB/1TB/No,Notebook,15.6,1366x768,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,Intel HD Graphics 520,No OS,1.9kg
3,1231,Dell,Inspiron 7579,2 in 1 Convertible,15.6,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,2.191kg
4,1020,HP,ProBook 640,Notebook,14.0,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,4GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.95kg


## 7. Replica el procesado en `test.csv`

> ⚠️ Usa `.transform()`, **nunca `.fit_transform()`** sobre los datos de test.
>
> Lo único que **no puedes hacer** es eliminar filas.

In [15]:
print("Forma de los datos listos:", X_pred_final.shape)
X_pred_final.head()

Forma de los datos listos: (391, 47)


,Inches,Weight,Ram_GB,Ops_IPS,Ops_FullHD,Cpu_GHz,SSD_GB,HDD_GB,Company_Apple,Company_Asus,...,OpSys_Windows 7,OpSys_macOS,Cpu_Brand_Intel Celeron/Pentium,Cpu_Brand_Intel Core i3,Cpu_Brand_Intel Core i5,Cpu_Brand_Intel Core i7,Cpu_Brand_Otros Intel,Gpu_Brand_ARM,Gpu_Brand_Intel,Gpu_Brand_Nvidia
laptop_ID,,,,,,,,,,,,,,,,,,,,,
209,15.6,2.400,16,0,1,2.8,512,0,False,False,...,False,False,False,False,False,True,False,False,False,True
1281,15.6,2.400,4,0,0,1.6,0,500,False,False,...,False,False,True,False,False,False,False,False,True,False
1168,15.6,1.900,4,0,0,2.0,0,1000,False,False,...,False,False,False,True,False,False,False,False,True,False
1231,15.6,2.191,8,1,1,2.5,256,0,False,False,...,False,False,False,False,True,False,False,False,True,False
1020,14.0,1.950,4,0,1,2.5,256,0,False,False,...,False,False,False,False,True,False,False,False,True,False


## 8. Genera la submission

### 8.1 ¿Qué formato espera Kaggle?

In [16]:
sample = pd.read_csv('./data/sample_submission.csv', encoding='latin-1')
sample.head()

,laptop_ID,Price_in_euros
0,209,1949.1
1,1281,805.0
2,1168,1101.0
3,1231,1293.8
4,1020,1832.6


### 8.2 Crea tu submission

In [17]:
final_predictions = final_model.predict(X_pred_final)

submission = pd.DataFrame({
    'laptop_ID': X_pred_raw['laptop_ID'],
    'Price_in_euros': final_predictions
})

submission.columns = sample.columns
submission.head()

,laptop_ID,Price_in_euros
0,209,1704.396140
1,1281,322.470960
2,1168,412.661535
3,1231,964.357256
4,1020,1096.005322


### 8.3 Chequeador

Pásale el chequeador antes de subir a Kaggle. Si todo está bien, guardará el CSV automáticamente con un nombre único.

In [18]:
def checker(df_to_submit, sample, filename=None):
    """
    Valida que tu submission tenga la forma requerida por Kaggle.
    Si es correcta, guarda el CSV listo para subir.
    Si no, lee el mensaje de error y corrígelo.
    """
    if df_to_submit.shape != sample.shape:
        print(' Shape incorrecto.')
        print(f'   Tu submission: {df_to_submit.shape} | Esperado: {sample.shape}')
        print('   Revisa que no hayas borrado filas del test ni añadido/quitado columnas.')
        return

    if not (df_to_submit.columns == sample.columns).all():
        print(' Nombres de columnas incorrectos.')
        print(f'   Tus columnas:       {list(df_to_submit.columns)}')
        print(f'   Columnas esperadas: {list(sample.columns)}')
        return

    if not (df_to_submit['laptop_ID'] == sample['laptop_ID']).all():
        print(' Los IDs no coinciden con sample_submission. Revisa que no hayas reordenado el test.csv.')
        return

    if filename is None:
        from datetime import datetime
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f'submission_{timestamp}.csv'

    df_to_submit.to_csv(filename, index=False)
    print(f" ¡Todo correcto! Submission guardada como '{filename}'. ¡A Kaggle!")

In [19]:
checker(submission, sample)

 ¡Todo correcto! Submission guardada como 'submission_20260703_115723.csv'. ¡A Kaggle!
